# 01 — Lagged cross-correlation screen

**docs/04 §4.1.** Kelp anomaly at quarter *t* against every environmental feature
anomaly at *t−0…4*, per polygon × environmental series. The first rung of the
docs/04 §4 analysis ladder, and the one that decides which relationships are
worth carrying to §4.3.

**This is a screen, and screening claims nothing.** docs/04 §5 makes §4.1
exploratory: it ranks candidates, it does not test them. So this notebook
reports correlation coefficients, sample sizes and an autocorrelation-adjusted
effective sample size — and **no p-values**, deliberately. The lag × feature ×
polygon grid generates hundreds of coefficients; a p-value column would invite
exactly the reading docs/04 §5 forbids.

**Reproducibility.** Runs top to bottom from `features/comparison.parquet` and
nothing else — no side reads of `observations/` or `raw/`. Nothing here is
stochastic, so there is no seed to set. Every table is stamped with the SHA-256
of the comparison file it was computed from; quote that digest in any figure
caption or write-up.

In [1]:
import hashlib
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 200)


def repo_root() -> Path:
    """The checkout root, so the notebook runs from anywhere."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "features" / "comparison.parquet").exists():
            return candidate
    raise FileNotFoundError(
        "no data/features/comparison.parquet above the working directory -- "
        "run `kelpcompare features` first"
    )


COMPARISON = repo_root() / "data" / "features" / "comparison.parquet"
DIGEST = hashlib.sha256(COMPARISON.read_bytes()).hexdigest()

comparison = pd.read_parquet(COMPARISON)

print(f"comparison.parquet   sha256:{DIGEST[:16]}   {len(comparison):,} rows")
print(f"Kelp Watch revision  {sorted(set(comparison['kelp_watch_revision']))}")
print(f"polygons             {comparison['polygon_id'].nunique()}")
print(f"lags                 {sorted(set(comparison['lag']))}")

comparison.parquet   sha256:7d2c62503276e7be   15,300 rows
Kelp Watch revision  [23]
polygons             6
lags                 [0, 1, 2, 3, 4]


## 1. The gate

`comparison.parquet` keeps every row, including those where either side is
unusable or where a lag reaches past the start of the environmental record —
docs/03 makes `usable` the single gate so that what filtering costs is visible
rather than already spent.

Spending it here, once, and reporting what it cost.

In [2]:
KELP = "kelp_area_m2_anom"
ENV_FEATURES = [
    column
    for column in comparison.columns
    if column.endswith("_anom") and column not in {KELP, "n_cells_kelp_anom"}
]

usable = (
    comparison["kelp_usable"] & comparison["env_usable"].fillna(False) & comparison[KELP].notna()
)
gated = comparison[usable]

attrition = pd.Series(
    {
        "all rows": len(comparison),
        "kelp side usable": int(comparison["kelp_usable"].sum()),
        "...and an environmental row exists": int(
            (comparison["kelp_usable"] & comparison["env_usable"].notna()).sum()
        ),
        "...and it is usable too": int(
            (comparison["kelp_usable"] & comparison["env_usable"].fillna(False)).sum()
        ),
        "...and the kelp anomaly is not null": len(gated),
    },
    name="rows",
)
print(attrition.to_string())
print()
missing_env = comparison["kelp_usable"] & comparison["env_usable"].isna()
env_start = int(comparison.loc[comparison["env_usable"].notna(), "env_year"].min())
unreachable = int((missing_env & (comparison["year"] < env_start)).sum())

print()
print(f"{int(missing_env.sum()):,} kelp-usable rows have no environmental row at all, and")
print(f"{unreachable:,} of those are kelp quarters before {env_start}, which no lag could have")
print(f"reached -- the kelp record starts in {int(comparison['year'].min())}, the environmental")
print(
    f"one in {env_start}. Only {int(missing_env.sum()) - unreachable:,} are the lag itself reaching"
)
print("back past the start of the environmental record.")

all rows                               15300
kelp side usable                       14415
...and an environmental row exists      6786
...and it is usable too                 6006
...and the kelp anomaly is not null     6006


7,629 kelp-usable rows have no environmental row at all, and
7,395 of those are kelp quarters before 2007, which no lag could have
reached -- the kelp record starts in 1984, the environmental
one in 2007. Only 234 are the lag itself reaching
back past the start of the environmental record.


## 2. Which features are worth screening at all

Two different reasons a feature should not enter the screen, and they must not be
confused with each other.

**Not applicable.** `quarterly_env` is wide and sparse by design (docs/03): a row
carries whichever features its `feature_set` defines and null elsewhere. Only
`sea_water_temperature` gets the docs/04 §2 ecological set, so
`air_temperature × days_above_20c` is not a weak feature — it is not a feature.

**Degenerate.** A feature whose values are pinned to a sensor's resolution floor
produces anomalies that encode nothing, and correlating them generates
coefficients that look strong and mean nothing.

That second one is not hypothetical here. **Quarterly minimum wind speed at
LJAC1 takes two values across the whole record** — `0.0` in 51 quarters and
`0.1` in 17 — because the anemometer cannot resolve below that. Its anomaly
takes four, being those two values against differing quarterly baselines, which
is the number the audit below counts; either way it is a disguised binary flag,
and in an earlier pass of this screen it
produced the largest coefficients in the entire grid, |r| up to 0.74, on an
effective sample size below 10.

The rule applied below, stated so it can be argued with:

- **not applicable** — no non-null values at all;
- **excluded** — fewer than 10 distinct anomaly values;
- **flagged `low_resolution`** — fewer than 25, kept but not to be read alone.

Distinct-value count rather than variance, because a genuine count feature like
`days_above_23c` has a legitimately small range and must not be thrown out for
it.

In [3]:
EXCLUDE_BELOW = 10
FLAG_BELOW = 25

audit = []
for parameter, block in gated.groupby("parameter"):
    for column in ENV_FEATURES:
        values = block[column].dropna()
        audit.append(
            {
                "parameter": parameter,
                "feature": column.removesuffix("_anom"),
                "present": len(values),
                "distinct": values.round(6).nunique(),
            }
        )
resolution = pd.DataFrame(audit)


def verdict(row) -> str:
    if row["present"] == 0:
        return "not applicable"
    if row["distinct"] < EXCLUDE_BELOW:
        return "excluded"
    return "low_resolution" if row["distinct"] < FLAG_BELOW else "ok"


resolution["verdict"] = resolution.apply(verdict, axis=1)

print(resolution["verdict"].value_counts().to_string())
print()
print("Not applicable -- the feature set does not define these for this parameter:")
na = resolution[resolution["verdict"] == "not applicable"]
print("   ", ", ".join(sorted(set(na["parameter"]))), "x the temperature feature set")
print()
print("Degenerate or low resolution:")
print(
    resolution[resolution["verdict"].isin(["excluded", "low_resolution"])]
    .sort_values("distinct")
    .to_string(index=False)
)

skip = set(
    zip(
        resolution.loc[resolution["verdict"].isin(["not applicable", "excluded"]), "parameter"],
        resolution.loc[resolution["verdict"].isin(["not applicable", "excluded"]), "feature"],
        strict=True,
    )
)

verdict
ok                20
not applicable    10
low_resolution     2
excluded           1

Not applicable -- the feature set does not define these for this parameter:
    air_temperature, wind_speed x the temperature feature set

Degenerate or low resolution:
            parameter        feature  present  distinct        verdict
           wind_speed            min     1482         4       excluded
           wind_speed            p05     1482        14 low_resolution
sea_water_temperature days_above_23c     2172        23 low_resolution


## 3. The screen

One row per polygon × environmental series × feature × lag.

**`n_eff` is the column to read, not `n`.** Quarterly anomaly series are
autocorrelated, so the number of paired quarters overstates how much independent
evidence a coefficient rests on. `n_eff` applies the Quenouille/Bartlett
adjustment for the correlation between two autocorrelated series —

$$n_{\text{eff}} = n\,\frac{1 - r_1^{(x)} r_1^{(y)}}{1 + r_1^{(x)} r_1^{(y)}}$$

— which is a rough correction, not an exact one. It is here in place of p-values
precisely because it degrades gracefully: where two series are strongly
autocorrelated it collapses, and a coefficient sitting on `n_eff` of 8 announces
itself as worthless without anyone having to interpret a significance threshold.

**The lag-1 term is measured across calendar-adjacent quarters only.** The rows
this screen keeps are not the quarters the calendar has: a cloud-gapped kelp
quarter and an unusable environmental one each leave a hole, and
`air_temperature` and `wind_speed` have no Q2 anomaly at all, so their series
steps Q3, Q4, Q1, Q3. Treating each of those steps as lag 1 would measure
persistence across a hole, which biases the autocorrelation down and `n_eff`
correspondingly up — the one direction a ceiling must not be wrong in. A gap
breaks the pair rather than being bridged across it, on the same reasoning that
makes an unobserved day break a threshold spell (docs/04 §2) and makes the
QARTOD neighbour tests withdraw at a gap rather than guess (docs/04 §1). The
cell below reports what that costs.

**Read it as a ceiling on independent evidence, not an estimate of it.** Two
things bound it from the optimistic side. It is *capped at `n`*, because the
expression is symmetric in the sign of the lag-1 product and an anti-correlated
pair would otherwise be credited with more independent quarters than it has
quarters — uncapped it runs to infinity at a product of −1. And it *sees lag 1
only*: the kelp anomaly stays autocorrelated well past one quarter — 0.42 to
0.73 at lag 1 across the six beds, and still 0.24 to 0.32 at lag 4 — so a
higher-order correction discounts the lag-4 cells further than this one does.
The two `days_below_14c` candidates in `notebooks/README.md` fall from 50.6 and
49.8 to 42.0 and 40.2 under a Bartlett sum truncated at *K* = ⌊*n*/4⌋ = 17 with
the (1 − *k*/*n*) taper, and those are the strongest cells in the screen.

Both Pearson and Spearman are reported. A large gap between them is a sign the
association is driven by a handful of points, which at this sample size is the
common failure.

In [4]:
MIN_PAIRS = 12
MIN_ADJACENT = 3


def quarter_index(years: pd.Series, quarters: pd.Series) -> pd.Series:
    """A running quarter number, so "the quarter before" is arithmetic, not position."""
    return years * 4 + (quarters - 1)


def lag1(values: pd.Series, quarters: pd.Series) -> float:
    """Lag-1 autocorrelation, over calendar-adjacent quarters only.

    The rows reaching here are not the quarters the calendar has. A cloud-gapped
    kelp quarter and an unusable environmental one each leave a hole, and
    `air_temperature` and `wind_speed` have no Q2 anomaly at all, so their series
    steps Q3, Q4, Q1, Q3. A positional shift would call each of those steps lag 1
    and measure persistence across a hole -- which biases the autocorrelation
    down, and so `n_eff` up, in the one direction a ceiling must not be wrong in.

    So a gap breaks the pair rather than being bridged across, exactly as an
    unobserved day breaks a threshold spell (docs/04 s2), and where too few
    adjacent pairs survive this withdraws rather than guessing, as the QARTOD
    neighbour tests do (docs/04 s1). `effective_n` reads that withdrawal as
    "unknown" and applies no discount, which is the optimistic direction -- but
    an honest unknown beats a number measured across a hole.
    """
    adjacent = quarters.diff() == 1
    if int(adjacent.sum()) < MIN_ADJACENT:
        return float("nan")
    return values[adjacent].corr(values.shift(1)[adjacent])


def effective_n(n: int, r1x: float, r1y: float) -> float:
    """Quenouille/Bartlett adjustment for two autocorrelated series, capped at `n`.

    The cap is not cosmetic. The expression is symmetric in the sign of the
    lag-1 product, so a negative product inflates where a positive one
    discounts, without bound and with a pole at -1. An effective sample larger
    than the number of quarters is the correction running backwards:
    persistence can only cost independent observations, never buy them, so `n`
    is the ceiling and a product at or below zero buys nothing back.
    """
    if pd.isna(r1x) or pd.isna(r1y):
        return float(n)
    product = r1x * r1y
    if product <= 0:
        return float(n)
    return max(1.0, float(n) * (1 - product) / (1 + product))


rows = []
steps = bridged = 0
fewest_adjacent = None
for (polygon, site, parameter, lag), block in gated.groupby(
    ["polygon_id", "site_id", "parameter", "lag"], sort=True
):
    ordered = block.sort_values(["year", "quarter"])
    for column in ENV_FEATURES:
        feature = column.removesuffix("_anom")
        if (parameter, feature) in skip:
            continue
        pair = (
            ordered[["year", "quarter", KELP, column]]
            .dropna(subset=[KELP, column])
            .reset_index(drop=True)
        )
        if len(pair) < MIN_PAIRS:
            continue
        quarters = quarter_index(pair["year"], pair["quarter"])
        adjacent = int((quarters.diff() == 1).sum())
        steps += len(pair) - 1
        bridged += (len(pair) - 1) - adjacent
        fewest_adjacent = adjacent if fewest_adjacent is None else min(fewest_adjacent, adjacent)
        rows.append(
            {
                "polygon_id": polygon,
                "site_id": site,
                "parameter": parameter,
                "feature": feature,
                "lag": int(lag),
                "n": len(pair),
                "n_eff": round(
                    effective_n(
                        len(pair),
                        lag1(pair[KELP], quarters),
                        lag1(pair[column], quarters),
                    ),
                    1,
                ),
                "pearson_r": round(pair[KELP].corr(pair[column]), 3),
                "spearman_rho": round(pair[KELP].corr(pair[column], method="spearman"), 3),
            }
        )

screen = pd.DataFrame(rows).merge(
    resolution[["parameter", "feature", "distinct", "verdict"]],
    on=["parameter", "feature"],
    how="left",
)
screen["low_resolution"] = screen["verdict"].eq("low_resolution")
screen = screen.drop(columns=["verdict"])

print(
    f"{len(screen):,} cells over {screen['polygon_id'].nunique()} polygons, "
    f"{screen['feature'].nunique()} features, {screen['lag'].nunique()} lags"
)
print(f"median n {screen['n'].median():.0f}  ->  median n_eff {screen['n_eff'].median():.0f}")
print()
print(
    f"{bridged:,} of {steps:,} steps between consecutive screened rows ({100 * bridged / steps:.0f}%)"
)
print("cross a quarter gap, and are excluded from the lag-1 autocorrelation rather than")
print(f"counted as adjacent. The thinnest cell still keeps {fewest_adjacent} adjacent pairs, so no")
print(f"cell falls back to the no-discount branch for want of {MIN_ADJACENT} of them.")

660 cells over 6 polygons, 11 features, 5 lags
median n 60  ->  median n_eff 46

5,724 of 39,354 steps between consecutive screened rows (15%)
cross a quarter gap, and are excluded from the lag-1 autocorrelation rather than
counted as adjacent. The thinnest cell still keeps 30 adjacent pairs, so no
cell falls back to the no-discount branch for want of 3 of them.


## 4. The matrix

docs/04 §4.1 asks for a lag–feature correlation matrix. Here it is for the pair
the project is actually about: **the La Jolla bed against the water temperature
at LJAC1**, the station inside it.

What to look for, per docs/04 §4.1, is whether the *known physics* shows up —
heat stress at short lags, a cold-water/nitrate association at longer ones. What
to remember is that this is one cell of a grid of several hundred, and that
|r| ≈ 0.2 on ~70 quarters is a hint, not a finding.

In [5]:
def matrix(polygon: str, parameter: str, statistic: str = "pearson_r") -> pd.DataFrame:
    """The lag x feature matrix for one polygon-series pair."""
    cell = screen[(screen["polygon_id"] == polygon) & (screen["parameter"] == parameter)]
    return cell.pivot_table(index="feature", columns="lag", values=statistic)


print("La Jolla kelp area anomaly vs LJAC1 sea water temperature -- Pearson r")
print(f"(comparison sha256:{DIGEST[:16]})")
print()
print(matrix("KELP:LA-JOLLA", "sea_water_temperature").to_string())

La Jolla kelp area anomaly vs LJAC1 sea water temperature -- Pearson r
(comparison sha256:7d2c62503276e7be)

lag                           0      1      2      3      4
feature                                                    
days_above_20c           -0.245 -0.198  0.044  0.084  0.059
days_above_23c           -0.067 -0.256 -0.150 -0.019  0.104
days_below_14c            0.173  0.003 -0.073  0.252  0.277
degree_days_above_18c    -0.150 -0.241 -0.034  0.062  0.063
max                      -0.184 -0.160  0.077  0.014 -0.132
max_spell_above_20c_days -0.097 -0.168 -0.001  0.035  0.114
mean                     -0.203 -0.170  0.048 -0.045 -0.113
min                      -0.069 -0.095 -0.004 -0.034 -0.206
p05                      -0.146 -0.149  0.015 -0.066 -0.156
p95                      -0.187 -0.168  0.043 -0.034 -0.123
variance                 -0.114 -0.090  0.013  0.019  0.003


## 5. Candidates, ranked

Sorted by |r|, with the two columns that decide whether a coefficient deserves a
second look: `n_eff`, and whether Pearson and Spearman agree.

**Nothing here is a result.** These are the rows to argue about when choosing
what to pre-register in `notebooks/README.md` and carry into docs/04 §4.3.

In [6]:
ranked = (
    screen.assign(
        abs_r=screen["pearson_r"].abs(),
        rank_gap=(screen["pearson_r"] - screen["spearman_rho"]).abs().round(3),
    )
    .sort_values("abs_r", ascending=False)
    .drop(columns=["abs_r", "site_id"])
)

print("Strongest 15 associations in the screen:")
print(ranked.head(15).to_string(index=False))
print()
print("Strongest 10 resting on an effective sample of at least 30:")
print(ranked[ranked["n_eff"] >= 30].head(10).to_string(index=False))

Strongest 15 associations in the screen:
         polygon_id             parameter        feature  lag  n  n_eff  pearson_r  spearman_rho  distinct  low_resolution  rank_gap
     KELP:ENCINITAS sea_water_temperature days_below_14c    4 71   50.6      0.424         0.393        51           False     0.031
KELP:IMPERIAL-BEACH            wind_speed       variance    2 50   50.0     -0.380        -0.419        50           False     0.039
KELP:IMPERIAL-BEACH       air_temperature            p95    2 49   49.0      0.380         0.041        45           False     0.339
  KELP:SOLANA-BEACH       air_temperature            p05    4 47   20.6     -0.349        -0.320        40           False     0.029
  KELP:SOLANA-BEACH            wind_speed            p05    3 49   19.5      0.348         0.365        14            True     0.017
     KELP:ENCINITAS       air_temperature            p05    4 47   20.3     -0.347        -0.288        40           False     0.059
  KELP:SOLANA-BEACH sea_wate

## 6. What this does not say

Carried from docs/04 §6 and the `analysis-review` skill, because a screen is
where over-reading starts:

- **Landsat canopy is a surface expression.** Urchin grazing, subsurface
  condition and predator dynamics are invisible to every dataset in this system,
  so unexplained variance here is ecological rather than statistical.
- **Cloud-gap missingness leans winter** — 9.1% of Q4 and 5.8% of Q1 quarters
  have no cloud-free observation against 0.8% of Q3 — so a screen run on
  non-null quarters is weighted away from winter.
- **Temperature and nitrate proxies are anti-correlated by regional
  oceanography.** A coefficient on a warm feature cannot be read as isolating
  thermal stress from nutrient limitation. That separation is not available from
  this data at all.
- **The 2007–2019 baseline contains the 2014–2016 marine heatwave** on both
  sides, so warm anomalies are damped by the event the analysis most wants to
  detect (docs/04 §3).
- **`air_temperature` and `wind_speed` have no Q2 anomaly at any polygon** —
  their Q2 baseline holds 9 years against the 10-year minimum
  ([#30](https://github.com/cweber12/kelp-compare/issues/30)). Their `n` is
  correspondingly ~50 where sea water temperature has ~73.
- **The project's key question is not answered here.** docs/04 §4.5 compares a
  project sensor against a public station for kelp at increasing distance; it
  needs polygon geometry, which `polygons.geojson` records as null for all six
  beds, and the yellow buoy's position, which `sites.json` records as unverified.